# Served vs unserved demand (Huff), quick first cut

**Question:** of each cell's addressable women, how many does each premium supplier (our lounges and the premium salons) serve, and how many does nobody serve?

**Model** (`src/model/served.py`):
- **Who can reach a cell:** every premium supplier within 20 minutes of it. The bands are 10, 15 and 20 minutes. They come from the cell's own Mapbox isochrones where we have them, and from straight-line km bands fitted on those cells elsewhere.
- **How the cell's women are split:** share = weight ÷ (sum of weights + outside option), where weight = reviews^ALPHA × DECAY[band].
- **What's unserved:** the outside option takes the rest, i.e. women who go elsewhere or use no premium salon.
- **Leave-one-out:** close each lounge, re-split, and record how much of its served demand the siblings take over (`absorbed`).

**Status:** the parameters below are starting points, not calibrated. This notebook shows what they produce, so we can decide together.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_rows", 80)
from src.baseline import run
from src.data_v3 import load_v3
from src.config import REPO_ROOT, load_baseline_assumptions
from src.model import served as S, scorecard, growth
r = run(); v3 = load_v3(); a = load_baseline_assumptions(REPO_ROOT / "data/scenarios/baseline.yaml")
old = pd.read_csv(REPO_ROOT / "data/scenarios/baseline_v3_labels.csv")
old_action = dict(zip(old.id, old.action))

## 1. Parameters and reach

In [2]:
acc = S._ACCESS[id(v3)]
print("ALPHA", S.ALPHA, "| DECAY", S.DECAY, "| OUTSIDE_PCTL", S.OUTSIDE_PCTL)
print("cell x supplier pairs within 20 min:", len(acc[1]), acc[1].source.value_counts().to_dict())
print("straight-line band edges (km), fitted as the 75th pct of isochrone-reached pairs:", {k: round(x, 1) for k, x in acc[2].items()})

ALPHA 1.0 | DECAY {10: 1.0, 15: 0.5, 20: 0.25} | OUTSIDE_PCTL 50
cell x supplier pairs within 20 min: 106547 {'isochrone': 92813, 'straight_line': 13734}
straight-line band edges (km), fitted as the 75th pct of isochrone-reached pairs: {10: 4.9, 15: 8.7, 20: 12.7}


## 2. Where the UAE's addressable women go

In [3]:
from src.features.lounges import _SERVED
sv = next(iter(_SERVED.values()))
cells = sv["cells"].join(v3.cells[["emirate"]])
t = cells.groupby("emirate")[["demand", "served_lounges", "served_competitors", "unserved"]].sum()
t.loc["UAE"] = t.sum()
(t.div(t.demand, axis=0)[["served_lounges", "served_competitors", "unserved"]] * 100).round(1).assign(demand=t.demand.round(-3))

,served_lounges,served_competitors,unserved,demand
emirate,,,,
Abu Dhabi,13.5,53.2,33.4,954000.0
Ajman,1.4,79.7,18.9,154000.0
Dubai,1.7,87.5,10.8,1000000.0
Fujairah,3.2,45.4,51.4,92000.0
Ras Al Khaimah,5.9,60.5,33.6,145000.0
Sharjah,2.1,71.7,26.1,600000.0
Umm Al Quwain,0.0,60.6,39.4,34000.0
UAE,5.8,70.0,24.2,2978000.0


## 3. Per lounge: served demand, absorption, and v3's estimate

In [4]:
f = pd.DataFrame([dict(lounge=x.branch_id, emirate=x.emirate, reviews=x.review_count, served=x.served_women,
                       absorbed=x.absorbed_share, v3_est_customers=x.est_customers, catchment_women=x.addressable_women)
                  for x in r.features if not x.not_scored]).set_index("lounge")
print("Spearman served vs reviews:", round(f.served.corr(f.reviews, method="spearman"), 2),
      "| served vs v3 est_customers:", round(f.served.corr(f.v3_est_customers, method="spearman"), 2))
f.sort_values("served", ascending=False).round(2)

Spearman served vs reviews: 0.45 | served vs v3 est_customers: 0.92


,emirate,reviews,served,absorbed,v3_est_customers,catchment_women
lounge,,,,,,
khalifa-city-a,Abu Dhabi,2465,19353.24,0.22,11486.57,91431.59
al-ain,Abu Dhabi,1332,17289.94,0.00,12617.50,102438.95
al-dhafra,Abu Dhabi,359,15201.94,0.00,16246.70,22927.30
shakhbout-city,Abu Dhabi,1277,13534.77,0.19,12239.50,134563.91
shahama,Abu Dhabi,757,12953.05,0.20,5725.09,65769.16
baniyas,Abu Dhabi,697,11457.51,0.14,9999.47,106825.82
al-falah,Abu Dhabi,951,10402.91,0.17,27054.29,49946.83
ras-al-khaimah,Ras Al Khaimah,772,8531.25,0.00,7713.24,55839.70
al-maqta,Abu Dhabi,1295,7090.61,0.18,6714.93,126053.63


## 4. Sensitivity: how the parameters move served demand

In [5]:
from src.market import women_15plus, affluence_weight
w = women_15plus(v3.cells, v3.emirates, 0.055); aw = w * affluence_weight(v3.cells.rent_observed, w, 0.5)
ns = frozenset({"zayed-international-airport"}); base = (S.ALPHA, dict(S.DECAY), S.OUTSIDE_PCTL); rows = []
for alpha in (0.5, 1.0):
    for pct in (25, 50, 75):
        for decay in ({10: 1, 15: 0.5, 20: 0.25}, {10: 1, 15: 0.7, 20: 0.5}):
            S.ALPHA, S.DECAY, S.OUTSIDE_PCTL = alpha, decay, pct
            o = S.served(v3, a, aw, not_scored=ns); c = o["cells"]
            s = pd.Series(o["served"])
            rows.append(dict(alpha=alpha, outside_pct=pct, decay15_20=f"{decay[15]}/{decay[20]}",
                             unserved_pct=round(100 * c.unserved.sum() / c.demand.sum(), 1),
                             lounges_pct=round(100 * c.served_lounges.sum() / c.demand.sum(), 1),
                             rank_corr_vs_base=round(s.corr(f.served, method="spearman"), 2),
                             top3=", ".join(s.sort_values(ascending=False).index[:3])))
S.ALPHA, S.DECAY, S.OUTSIDE_PCTL = base
pd.DataFrame(rows)

,alpha,outside_pct,decay15_20,unserved_pct,lounges_pct,rank_corr_vs_base,top3
0,0.5,25,0.5/0.25,23.0,3.8,0.95,"al-dhafra, al-ain, shahama"
1,0.5,25,0.7/0.5,21.4,3.9,0.95,"al-dhafra, al-ain, shahama"
2,0.5,50,0.5/0.25,24.4,3.7,0.95,"al-dhafra, al-ain, shahama"
3,0.5,50,0.7/0.5,22.7,3.8,0.94,"al-dhafra, al-ain, shahama"
4,0.5,75,0.5/0.25,26.3,3.6,0.95,"al-dhafra, al-ain, shahama"
5,0.5,75,0.7/0.5,24.3,3.7,0.95,"al-dhafra, al-ain, khalifa-city-a"
6,1.0,25,0.5/0.25,21.8,6.0,1.00,"khalifa-city-a, al-dhafra, al-ain"
7,1.0,25,0.7/0.5,20.6,6.1,0.98,"khalifa-city-a, al-dhafra, al-ain"
8,1.0,50,0.5/0.25,24.2,5.8,1.00,"khalifa-city-a, al-ain, al-dhafra"
9,1.0,50,0.7/0.5,22.7,5.9,0.98,"khalifa-city-a, al-ain, al-dhafra"


## 5. Open questions for review

1. **The results hold up across the parameters** (section 4). Over all 12 combinations, unserved demand stays at 21–28% and lounge rankings barely move (Spearman ≥ 0.94 against the base). α moves our lounges' total share most (about 3.7% vs 6.0%). The outside option still needs a meaning: women who don't use premium salons at all, or women willing to travel beyond 20 minutes?
2. **Cells nobody reaches count as 100% unserved.** Remote, empty places therefore look like opportunities (see `growth.ipynb`). Should demand far from *any* salon count, or is it "no market" rather than "unserved"?
3. **Reviews are the attractiveness term.** Big Dubai competitors (thousands of reviews) take most of each Dubai cell, so four of the five Dubai lounges serve 1.7k–2.8k women (mirdif-35 serves 6.5k). That's plausible but harsh. Should α be lower, or should attractiveness be capped?
5. **Served demand tracks v3's unscored `est_customers` closely** (Spearman 0.92). The new demand signal is close to the "estimated customers" figure v3 computed but chose not to score.
4. **There's no capacity cap yet.** khalifa-city-a serves about 19k. Is that within what one lounge can handle?